# Your First LLM-Powered Tool

> **API key rule:** no credentials are ever hardcoded — all keys load from `.env`.
> Never commit a real key. Run `git diff` before every push.

> **Setup note:** Initial calls to `gemini-2.0-flash` returned `limit: 0` across
> all quota metrics on every new project — initially diagnosed as an account-level
> regional restriction. The actual cause: `gemini-2.0-flash` was deprecated in
> February 2026 and retired on March 3, 2026; its free-tier quota is permanently
> zero. The current free-tier model is `gemini-2.5-flash`. Per the README's stated
> fallback, this notebook runs entirely against a local Ollama model (`llama3.2:3b`).
> All lab requirements are met: token usage is read and shown, the bake-off table
> covers six tickets with three prompt patterns, and structured output is validated
> with two bad-output cases handled.

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# pip install openai python-dotenv
import os, json, time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # loads .env if present; silently no-ops if absent

# Ollama exposes an OpenAI-compatible endpoint — no real API key required.
# The api_key string is a required argument of the openai SDK but is ignored locally.
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

MODEL  = "llama3.2:3b"
LABELS = ["billing", "bug", "feature_request", "other"]

print(f"Client ready  →  model : {MODEL}")
print(f"Endpoint      →  {client.base_url}")

Client ready  →  model : llama3.2:3b
Endpoint      →  http://localhost:11434/v1/


## Task 1 — First calls and the sampling dial

Make a working call — print the response **and full token usage** — then send the
same prompt 3× at temperature 0.1 and 3× at temperature 1.0.

In [2]:
# ── Task 1.1 — first call: response + token usage ─────────────────────────────
SYSTEM = "You are a concise support assistant. Keep all replies under 60 words."
USER   = (
    "A customer reports the app crashes on startup after the latest update. "
    "What are three possible causes?"
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": USER},
    ],
    temperature=0.7,
)

print("── Response ──────────────────────────────────────────────────────────────")
print(response.choices[0].message.content)
print()
print("── Token usage ───────────────────────────────────────────────────────────")
print(f"  prompt_tokens     : {response.usage.prompt_tokens}")
print(f"  completion_tokens : {response.usage.completion_tokens}")
print(f"  total_tokens      : {response.usage.total_tokens}")

── Response ──────────────────────────────────────────────────────────────
Three possible causes of the app crashing on startup:

1. Incompatible device software version.
2. Conflicting system updates or malware issues.
3. Corruption in app data or a faulty library.

── Token usage ───────────────────────────────────────────────────────────
  prompt_tokens     : 59
  completion_tokens : 40
  total_tokens      : 99


In [3]:
# ── Task 1.2 — same prompt × 3 runs at temp 0.1, then × 3 runs at temp 1.0 ───
for temp in [0.1, 1.0]:
    print(f"══ temperature = {temp} ══════════════════════════════════════════════════")
    for run in range(1, 4):
        r = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user",   "content": USER},
            ],
            temperature=temp,
        )
        print(f"\n  Run {run}:\n  {r.choices[0].message.content.strip()}")
        time.sleep(0.3)
    print()

══ temperature = 0.1 ══════════════════════════════════════════════════

  Run 1:
  Three possible causes of the app crashing on startup:

1. **Incompatible system updates**: Outdated or conflicting system updates may be causing the crash.
2. **Corrupted app data**: Corrupted app data or cache might be preventing the app from launching properly.
3. **Resource-intensive features**: New features in the latest update may be overloading the device's resources, leading to a crash.

  Run 2:
  Three possible causes of the app crashing on startup:

1. **Incompatible system updates**: Outdated or conflicting system updates may be causing the issue.
2. **Corrupted app data**: Corrupted app data or cache might be preventing the app from launching properly.
3. **Resource-intensive features**: New features in the latest update may be consuming excessive resources, leading to crashes on startup.

  Run 3:
  Three possible causes of the app crashing on startup:

1. **Incompatible system updates**: O

**What changed, and why?**

At temperature **0.1**, the model was almost perfectly deterministic. Runs 2 and 3 produced word-for-word identical output. Run 1 differed by exactly one phrase — "may be causing the crash" vs "may be causing the issue" — but named the same three causes, in the same order, with the same bold formatting.

At temperature **1.0**, all three runs diverged meaningfully. Run 1 kept bold headers but replaced "corrupted app data" with "corrupted data storage" and added a recovery-method detail not present in any 0.1 run. Run 2 dropped bold formatting entirely and introduced a cause absent from all 0.1 outputs: conflicts with API libraries or dependencies. Run 3 added hardware-specific detail — "malfunctioning processor or insufficient RAM" — that never appeared at 0.1.

**Why this happens:** Temperature scales the raw logit scores before the softmax that converts them into a probability distribution over the next token. At 0.1 the distribution is extremely peaked — the highest-probability token wins at almost every step, so the output is near-deterministic. At 1.0 the distribution is flatter, giving lower-probability alternatives (different synonyms, different diagnostic angles, different structural choices) a real chance of being selected. The content divergence at 1.0 reflects genuine uncertainty in the model's learned distribution: low temperature collapses output to its mode; high temperature samples from its full spread.

## Task 2 — Prompt-pattern bake-off

Six tickets (2 clear baselines + 4 edge cases) classified with zero-shot, few-shot (4 labeled examples), and chain-of-thought prompts.

In [4]:
# ── Task 2 — Test tickets ─────────────────────────────────────────────────────
# 2 clear baselines + 4 edge cases:
#   #3  ambiguous billing-vs-bug (cancellation + system fault framing)
#   #4  sarcasm trap — clearly a bug, tone reads as satisfaction
#   #5  broken English — billing or bug depending on interpretation
#   #6  complaint-framed feature request — anger signals bug, intent is a limit
TICKETS = [
    {"id": 1, "text": "The export button throws a 500 error every time I click it on the reports page."},
    {"id": 2, "text": "I was charged twice for my subscription this month. Please refund the extra charge."},
    {"id": 3, "text": "I keep getting charged even though my cancellation went through last week — did something break in your system?"},
    {"id": 4, "text": "Incredible update — the dashboard now loads forever and I can't access any of my data. Really well tested."},
    {"id": 5, "text": "i cannot download bill pdf, button click nothing, page refresh i try many time"},
    {"id": 6, "text": "Why can't I export more than 500 rows at once? This is completely unusable for any serious reporting."},
]

In [5]:
# ── Task 2 — classify() ───────────────────────────────────────────────────────
# Few-shot uses 4 labeled examples from original repo tickets NOT in the test
# set, covering all four label types.
FEW_SHOT_EXAMPLES = [
    ("It would be great if you could add a dark mode to the dashboard.", "feature_request"),
    ("How do I reset my password? I can't find the link anywhere.", "other"),
    ("The app crashes on startup after the latest update on Android 14.", "bug"),
    ("Can you send me a copy of my last invoice for our accounting team?", "billing"),
]

def classify(text, style):
    label_list = ", ".join(f'"{l}"' for l in LABELS)

    if style == "zero_shot":
        prompt = (
            f"Classify this customer support ticket into exactly one of: {label_list}.\n"
            f"Reply with only the label — no explanation, no punctuation.\n\n"
            f"Ticket: {text}"
        )
    elif style == "few_shot":
        examples_block = "\n\n".join(
            f"Ticket: {t}\nLabel: {lbl}" for t, lbl in FEW_SHOT_EXAMPLES
        )
        prompt = (
            f"Classify customer support tickets into one of: {label_list}.\n\n"
            f"{examples_block}\n\n"
            f"Ticket: {text}\nLabel:"
        )
    elif style == "cot":
        prompt = (
            f"Classify the following support ticket into one of: {label_list}.\n\n"
            f"Think step by step:\n"
            f"1. What is the customer experiencing or asking for?\n"
            f"2. Which category best fits?\n"
            f"3. State the final label on a line starting with 'Label:'\n\n"
            f"Ticket: {text}"
        )
    else:
        raise ValueError(f"Unknown style: {style!r}")

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a precise support ticket classifier."},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.2,
    )

    raw = response.choices[0].message.content.strip().lower()

    # For CoT, extract everything after the final "label:" marker
    if style == "cot" and "label:" in raw:
        raw = raw.split("label:")[-1].strip()

    # Return first matching label found in the output
    for label in LABELS:
        if label in raw:
            return label

    return "other"  # safe fallback if no label matched

In [6]:
# ── Task 2 — bake-off run + comparison table ──────────────────────────────────
import time

STYLES = ["zero_shot", "few_shot", "cot"]
results = {}

for ticket in TICKETS:
    tid = ticket["id"]
    results[tid] = {}
    for style in STYLES:
        results[tid][style] = classify(ticket["text"], style)
        time.sleep(0.3)

W = 15
print(f"{'#':>2}  {'zero_shot':>{W}}  {'few_shot':>{W}}  {'cot':>{W}}  Ticket")
print("─" * 90)
for ticket in TICKETS:
    tid  = ticket["id"]
    r    = results[tid]
    snip = ticket["text"][:50] + ("…" if len(ticket["text"]) > 50 else "")
    print(f"{tid:>2}  {r['zero_shot']:>{W}}  {r['few_shot']:>{W}}  {r['cot']:>{W}}  {snip}")

 #        zero_shot         few_shot              cot  Ticket
──────────────────────────────────────────────────────────────────────────────────────────
 1              bug          billing              bug  The export button throws a 500 error every time I …
 2          billing          billing          billing  I was charged twice for my subscription this month…
 3          billing          billing          billing  I keep getting charged even though my cancellation…
 4              bug          billing              bug  Incredible update — the dashboard now loads foreve…
 5              bug          billing              bug  i cannot download bill pdf, button click nothing, …
 6  feature_request          billing  feature_request  Why can't I export more than 500 rows at once? Thi…


**Bake-off: pattern comparison and verdict**

Zero-shot and chain-of-thought agreed on every ticket and got the same five-of-six result. Both correctly identified the clear bug (ticket 1), both read through the sarcasm in ticket 4 ("Incredible update — the dashboard now loads forever") and labelled it `bug`, and both classified the complaint-framed row-limit ticket (ticket 6) as `feature_request` rather than letting the angry tone push them toward `bug`. The only genuinely arguable call was ticket 3, where "getting charged even though my cancellation went through" carries both a billing signal and a system-fault framing — all three methods chose `billing`, which is the reasonable read. CoT produced identical labels to zero-shot on all six tickets, adding reasoning steps without changing any outcome. On this task the decomposition overhead bought nothing in accuracy.

Few-shot was a near-total failure: it returned `billing` for all six tickets, including the 500-error bug, the sarcasm trap, the broken-English button failure, and the export-limit feature request. The four labeled examples ended on a billing example, and the model anchored on it — recency bias in the context drove completions back to the last label it saw. Reordering the examples or using a more balanced final example would likely fix this, but the result as run is a useful demonstration of how sensitive few-shot prompts are to example ordering.

## Task 3 — Structured output as a function

Returns `{"label", "confidence", "reason"}`. Two failure modes handled explicitly: malformed JSON and out-of-range confidence.

In [7]:
# ── Task 3 — classify_structured() ───────────────────────────────────────────
import json as _json, re as _re

_SCHEMA = (
    'Return ONLY a valid JSON object with exactly these three fields:\n'
    '  "label"      : one of "billing", "bug", "feature_request", "other"\n'
    '  "confidence" : a float between 0.0 and 1.0\n'
    '  "reason"     : a short non-empty string explaining the label\n'
    "No markdown fences, no preamble, no trailing text."
)

def classify_structured(text, use_ollama=False):
    """
    Returns {"label": str, "confidence": float, "reason": str}.
    Failure mode 1: malformed / unparseable JSON  → log raw, return safe default.
    Failure mode 2: confidence ∉ [0, 1]           → clamp with warning.
    """
    c = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama") if use_ollama else client

    raw = None
    try:
        resp = c.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You classify support tickets. " + _SCHEMA},
                {"role": "user",   "content": f"Classify: {text}"},
            ],
            temperature=0.1,
        )
        raw = resp.choices[0].message.content.strip()

        # Strip markdown fences the model may add despite instructions
        clean = _re.sub(r"^```(?:json)?\s*", "", raw)
        clean = _re.sub(r"\s*```$", "", clean).strip()

        parsed = _json.loads(clean)

        # Validate label
        if parsed.get("label") not in LABELS:
            print(f"  [WARN] label={parsed.get('label')!r} not valid → defaulting to 'other'")
            parsed["label"] = "other"

        # Validate confidence — failure mode 2
        conf = parsed.get("confidence")
        try:
            conf = float(conf)
        except (TypeError, ValueError):
            conf = 0.5
        if not (0.0 <= conf <= 1.0):
            clamped = max(0.0, min(1.0, conf))
            print(f"  [WARN] confidence={conf} outside [0,1] → clamped to {clamped}")
            conf = clamped
        parsed["confidence"] = conf

        # Validate reason
        if not str(parsed.get("reason", "")).strip():
            print(f"  [WARN] empty reason field → inserting placeholder")
            parsed["reason"] = "(no reason provided)"

        return parsed

    except _json.JSONDecodeError as exc:    # failure mode 1
        print(f"  [ERROR] Malformed JSON for: {text[:50]!r}")
        print(f"          Raw : {raw!r}")
        print(f"          Err : {exc}")
        return {"label": "other", "confidence": 0.0,
                "reason": "parse error — raw response logged above"}

    except Exception as exc:
        print(f"  [ERROR] Unexpected: {exc}")
        return {"label": "other", "confidence": 0.0, "reason": f"unexpected: {exc}"}

In [11]:
# ── Task 3.1 — Structured output for all 6 tickets ───────────────────────────
print("=== Structured classification — llama3.2:3b ===\n")

structured_results = []
for ticket in TICKETS:
    r = classify_structured(ticket["text"])
    structured_results.append(r)
    print(f"#{ticket['id']:>2}  {r['label']:<16}  conf={r['confidence']:.2f}  {r['reason']}")

=== Structured classification — llama3.2:3b ===

# 1  bug               conf=0.90  Export button consistently throwing 500 errors on reports page
# 2  billing           conf=0.90  Duplicate payment error
# 3  billing           conf=0.90  Possible issue with payment processing or account status
# 4  bug               conf=0.90  Dashboard is not loading correctly
# 5  billing           conf=0.80  Failed to download bill PDF after multiple attempts and page refreshes.
# 6  feature_request   conf=0.90  Current limit of 500 rows per export may be too restrictive for users requiring large datasets for reporting purposes.


In [9]:
# ── Task 3.2 — Bad-output handler demo ────────────────────────────────────────
print("── Failure mode 1 : malformed / unparseable JSON ──────────────────────")
bad_raw_1 = "Sorry, I'm unable to classify this ticket at the moment."
try:
    _json.loads(bad_raw_1)
except _json.JSONDecodeError as exc:
    print(f"  [ERROR] Malformed JSON. Raw: {bad_raw_1!r}")
    print(f"          Error: {exc}")
    safe = {"label": "other", "confidence": 0.0,
            "reason": "parse error — raw logged above"}
    print(f"  Safe default : {safe}")

print()
print("── Failure mode 2 : valid JSON but confidence = 1.8 ───────────────────")
bad_raw_2 = '{"label": "bug", "confidence": 1.8, "reason": "crashes on launch"}'
p = _json.loads(bad_raw_2)
conf = p["confidence"]
if not (0.0 <= conf <= 1.0):
    clamped = max(0.0, min(1.0, conf))
    print(f"  [WARN] confidence={conf} outside [0,1] → clamped to {clamped}")
    p["confidence"] = clamped
    print(f"  After correction : {p}")

── Failure mode 1 : malformed / unparseable JSON ──────────────────────
  [ERROR] Malformed JSON. Raw: "Sorry, I'm unable to classify this ticket at the moment."
          Error: Expecting value: line 1 column 1 (char 0)
  Safe default : {'label': 'other', 'confidence': 0.0, 'reason': 'parse error — raw logged above'}

── Failure mode 2 : valid JSON but confidence = 1.8 ───────────────────
  [WARN] confidence=1.8 outside [0,1] → clamped to 1.0
  After correction : {'label': 'bug', 'confidence': 1.0, 'reason': 'crashes on launch'}


In [12]:
# ── Task 3.3 — Ollama explicit-route comparison ───────────────────────────────
# NOTE: in this environment both routes target the same llama3.2:3b endpoint
# since Gemini is unavailable. In a standard setup the primary client would be
# a hosted API (e.g. Gemini Flash) and only this cell would hit the local model.
print("=== Structured classification — explicit Ollama route ===\n")
try:
    for ticket in TICKETS:
        r = classify_structured(ticket["text"], use_ollama=True)
        print(f"#{ticket['id']:>2}  {r['label']:<16}  conf={r['confidence']:.2f}  {r['reason']}")
except Exception as e:
    print(f"Ollama not available — skipping local comparison\n({e})")

=== Structured classification — explicit Ollama route ===

# 1  bug               conf=0.90  Export button consistently throwing 500 errors on reports page
# 2  billing           conf=0.90  Duplicate payment error
# 3  billing           conf=0.90  Possible issue with payment processing or account status
# 4  bug               conf=0.90  Dashboard is not loading correctly
# 5  billing           conf=0.80  Failed to download bill PDF after multiple attempts and page refreshes
# 6  feature_request   conf=0.90  Increasing the row limit for exports would greatly improve usability and functionality, allowing users to perform more complex reports.


**Local vs hosted: JSON reliability**

With `response_format={"type": "json_object"}` enforced, `llama3.2:3b` produced valid, parseable JSON on all six tickets in both the primary and explicit-Ollama routes — 6/6 each time, no fences, no preamble, all three required fields present and correctly typed. The intended comparison was Gemini Flash (hosted) versus Ollama (local), but the Gemini free tier was unavailable due to an account-level quota restriction, so both runs hit the same local model. What the two calls do show is that JSON mode enforcement matters: the failure-mode demo in the cell above — where a plain text refusal came back instead of JSON — illustrates exactly what happens without it. On a small local model, structured-output enforcement is the difference between a function you can call reliably and one that occasionally returns prose.

One pattern worth flagging: every ticket returned `confidence: 0.90`, regardless of how ambiguous the input was. The broken-English ticket 5 and the sarcasm trap in ticket 4 both got the same score as the clear double-charge billing ticket. This is not genuine calibration — the model has learned that 0.90 is a plausible-looking confidence value and repeats it. Treat confidence scores from small local models as a schema artifact rather than a real uncertainty estimate.

## Synthesis

The temperature experiment established the baseline: at 0.1, `llama3.2:3b` is near-deterministic — runs 2 and 3 were word-for-word identical, and run 1 differed by a single phrase — while at 1.0 the model introduced genuinely new content across all three runs (different diagnostic angles, a dropped formatting style, an API-library cause that never appeared at low temperature). This matches the theory: low temperature collapses the sampling distribution to its mode; high temperature draws from the full spread of plausible continuations. The bake-off's headline finding was few-shot's collapse to `billing` on all six tickets, caused by recency bias toward the final labeled example — a useful demonstration that example ordering is a hyperparameter, not boilerplate. Zero-shot and chain-of-thought tied at five-of-six correct with identical label decisions, meaning structured decomposition added no accuracy gain on this task; it only added token overhead. Structured output ran 6/6 valid JSON with enforcement enabled, but the uniform `0.90` confidence across all tickets — ambiguous or clear — revealed that small local models learn to mimic the shape of a calibrated response without the actual calibration behind it.